# 🎬 Chinese Drama Dubber — Google Colab Edition

**AI-powered dubbing pipeline** that translates and dubs Chinese/Asian drama videos into 18+ languages.

Everything runs in the cloud — **zero MacBook storage used!**

| Feature | Details |
|---------|--------|
| 🌐 Languages | Hindi, Tamil, Telugu, Bengali, English, Spanish, French, + 11 more |
| 🧠 AI Models | Groq Whisper + Llama-4-Scout (all FREE) |
| 🗣️ TTS | Microsoft Edge Neural TTS (FREE) |
| 💾 Storage | Google Drive (no local space needed) |
| 💰 Cost | $0 |

---

## Step 1: Mount Google Drive & Setup

In [ ]:
# Mount Google Drive — all files will be stored here
from google.colab import drive
drive.mount('/content/drive')

# Create output folder on Google Drive
import os
DRIVE_OUTPUT = '/content/drive/MyDrive/DubbedVideos'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'✅ Google Drive mounted! Output folder: {DRIVE_OUTPUT}')

## Step 2: Install Dependencies

In [ ]:
# Install required packages (Colab already has ffmpeg)
!pip install -q groq>=0.11.0 edge-tts>=6.1.0 yt-dlp indic-transliteration>=2.3.0 nest-asyncio>=1.5.0

# Apply nest_asyncio fix for Colab's event loop
import nest_asyncio
nest_asyncio.apply()

# Verify ffmpeg
!ffmpeg -version | head -1
!yt-dlp --version
print('\n✅ All dependencies installed!')

## Step 3: Clone the Dubber Agent

In [ ]:
# Clone the repo (or use existing)
REPO_DIR = '/content/chinese-drama-dubber'
if os.path.exists(REPO_DIR):
    import shutil
    shutil.rmtree(REPO_DIR)
!git clone https://github.com/Dipesh600/chinese-drama-dubber.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f'📁 Working directory: {os.getcwd()}')
print('✅ Repository cloned!')

## Step 3b: Apply Colab Fix (Event Loop Patch)

This patches the TTS engine to work with Colab's event loop.

In [ ]:
# Patch tts_engine.py — fix asyncio.run() for Colab
tts_file = os.path.join(REPO_DIR, 'tts_engine.py')
with open(tts_file, 'r') as f:
    content = f.read()

# Add nest_asyncio import and _run_async helper at the top
patch = '''"""TTS Engine v3 — Edge TTS with Colab/Jupyter event loop fix."""
import os, json, logging, asyncio, subprocess
logger = logging.getLogger(__name__)

# Fix for Google Colab — already has a running event loop
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

def _run_async(coro):
    """Run async coroutine, compatible with Colab and normal Python."""
    try:
        loop = asyncio.get_event_loop()
        if loop.is_running():
            return loop.run_until_complete(coro)
        else:
            return asyncio.run(coro)
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            return loop.run_until_complete(coro)
        finally:
            loop.close()

'''

# Replace the old header
old_header = '"""TTS Engine v3 — Edge TTS with sentence-level units, rate/pitch per character, overlap prevention."""\nimport os, json, logging, asyncio, subprocess\nlogger = logging.getLogger(__name__)'
content = content.replace(old_header, patch)

# Replace asyncio.run() calls with _run_async()
content = content.replace(
    'asyncio.run(asyncio.wait_for(_gen_async(text, voice, rate, pitch, raw), timeout=15.0))',
    '_run_async(asyncio.wait_for(_gen_async(text, voice, rate, pitch, raw), timeout=15.0))'
)
content = content.replace(
    'asyncio.run(asyncio.wait_for(edge_tts.Communicate(text, voice).save(raw), timeout=15.0))',
    '_run_async(asyncio.wait_for(edge_tts.Communicate(text, voice).save(raw), timeout=15.0))'
)

with open(tts_file, 'w') as f:
    f.write(content)

print('✅ TTS engine patched for Colab compatibility!')

## Step 4: Set Your Groq API Key

Get your **FREE** API key from: https://console.groq.com/keys

In [ ]:
# Option 1: Set directly (replace with your key)
# os.environ['GROQ_API_KEY'] = 'gsk_your_key_here'

# Option 2: Use Colab secrets (recommended - more secure)
try:
    from google.colab import userdata
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
    print('✅ API key loaded from Colab Secrets!')
except:
    # Fallback: prompt for key
    import getpass
    os.environ['GROQ_API_KEY'] = getpass.getpass('Enter your Groq API Key: ')
    print('✅ API key set!')

# Verify
key = os.environ.get('GROQ_API_KEY', '')
print(f'🔑 Key: {key[:8]}...{key[-4:]}' if len(key) > 12 else '❌ No key set!')

## Step 5: 🎬 Dub Your Video!

### Supported Languages
Hindi, Tamil, Telugu, Bengali, Marathi, Gujarati, Kannada, Malayalam, Urdu, English, Spanish, French, Portuguese, German, Japanese, Korean, Arabic, Turkish

In [ ]:
#@title 🎬 Dubbing Configuration { display-mode: "form" }

#@markdown ### Video Settings
VIDEO_URL = 'https://youtu.be/m_mJhZuM4Bk' #@param {type:"string"}
TARGET_LANGUAGE = 'Hindi' #@param ['Hindi', 'Tamil', 'Telugu', 'Bengali', 'Marathi', 'Gujarati', 'Kannada', 'Malayalam', 'Urdu', 'English', 'Spanish', 'French', 'Portuguese', 'German', 'Japanese', 'Korean', 'Arabic', 'Turkish']
SOURCE_LANGUAGE = 'zh' #@param ['zh', 'en', 'auto']

#@markdown ### Audio Settings
KEEP_BACKGROUND_MUSIC = True #@param {type:"boolean"}
VIDEO_DESCRIPTION = 'Chinese drama video' #@param {type:"string"}

print(f'🎬 Video:    {VIDEO_URL}')
print(f'🌐 Language: {TARGET_LANGUAGE}')
print(f'🎵 BG Music: {"Smart Ducking" if KEEP_BACKGROUND_MUSIC else "OFF"}')
print(f'📁 Output:   {DRIVE_OUTPUT}')
print('\n⏳ Starting dubbing pipeline...')
print('=' * 65)

In [ ]:
# Run the dubbing pipeline — output goes to Google Drive!
import sys, importlib
sys.path.insert(0, REPO_DIR)

# Force reload modules to pick up the patched tts_engine
for mod_name in list(sys.modules.keys()):
    if mod_name in ['orchestrator', 'director', 'preprocessor', 'translator',
                    'dialogue_writer', 'sentence_merger', 'voice_caster',
                    'tts_engine', 'assembler', 'romanizer', 'validator']:
        del sys.modules[mod_name]

from orchestrator import run_agent

result = run_agent(
    url=VIDEO_URL,
    target_lang=TARGET_LANGUAGE,
    source_lang=SOURCE_LANGUAGE,
    user_description=VIDEO_DESCRIPTION,
    output_dir=DRIVE_OUTPUT,
    preserve_bg=KEEP_BACKGROUND_MUSIC
)

if result['success']:
    print('\n' + '=' * 65)
    print(f'  ✅ DUBBING COMPLETE!')
    print(f'  📹 Video:      {result["video_path"]}')
    print(f'  📝 Subtitles:  {result["srt_path"]}')
    print(f'  🎭 Content:    {result.get("content_type", "?")}')
    print(f'  👥 Speakers:   {result.get("real_speaker_count", "?")}')
    print(f'  ⏱️ Time:       {result.get("processing_time", "?")}s')
    print(f'  💾 Size:       {result.get("size_mb", "?")}MB')
    print(f'  🗂️ Saved to:   Google Drive → DubbedVideos/')
    print('=' * 65)
else:
    print(f'\n❌ FAILED: {result.get("error", "Unknown error")}')
    print(f'   Work dir: {result.get("work_dir", "?")}')

## Step 6: Preview & Download

In [ ]:
# Play the dubbed video right here in Colab!
if result.get('success'):
    from IPython.display import Video, display, HTML
    video_path = result['video_path']

    # Show video player
    try:
        display(Video(video_path, embed=True, width=640))
    except:
        print(f'Video saved at: {video_path}')
        print('Open Google Drive → DubbedVideos to find your video!')

    # Show subtitles preview
    srt_path = result.get('srt_path', '')
    if srt_path and os.path.exists(srt_path):
        print('\n📝 Subtitle Preview (first 10 lines):')
        print('-' * 40)
        with open(srt_path, encoding='utf-8') as f:
            lines = f.readlines()[:15]
            print(''.join(lines))
else:
    print('No video to preview — dubbing did not succeed.')

In [ ]:
# Copy final output to a clean Google Drive folder
if result.get('success'):
    import shutil
    import time

    ts = time.strftime('%Y%m%d_%H%M%S')
    clean_name = f'dubbed_{TARGET_LANGUAGE}_{ts}'
    final_dir = os.path.join(DRIVE_OUTPUT, clean_name)
    os.makedirs(final_dir, exist_ok=True)

    # Copy video and subtitles
    video_final = os.path.join(final_dir, f'dubbed_{TARGET_LANGUAGE}.mp4')
    srt_final = os.path.join(final_dir, f'subtitles_{TARGET_LANGUAGE}.srt')

    shutil.copy2(result['video_path'], video_final)
    if os.path.exists(result.get('srt_path', '')):
        shutil.copy2(result['srt_path'], srt_final)

    print(f'✅ Clean output saved to Google Drive:')
    print(f'   📹 {video_final}')
    print(f'   📝 {srt_final}')
    print(f'\n📂 Find it in: Google Drive → DubbedVideos → {clean_name}/')
else:
    print('Nothing to copy.')

---
## 🔄 Dub Another Video

Just go back to **Step 5**, change the `VIDEO_URL` and `TARGET_LANGUAGE`, and run again!

---

### 💡 Tips
- **Colab Secrets**: Go to 🔑 icon in left sidebar → Add `GROQ_API_KEY` for secure storage
- **GPU not needed**: This pipeline uses CPU only (TTS + ffmpeg)
- **Storage**: All temp files stay on Colab, only final video goes to Drive
- **Rate limits**: If Groq rate limits, wait a few minutes and re-run